# 12 - Thực nghiệm Dò tìm Siêu tham số & Tối ưu Trọng số Refuted và Label Smoothing
### Khảo sát Hàm Mất mát Phán quyết $\mathcal{L}_{\text{verdict}}$ với Cost-Sensitive Class Weighting và Label Smoothing

**Mục tiêu Khoa học & Đóng góp Nghiên cứu**:
1. **Cost-Sensitive Learning cho Fact-Checking**: Trong bài toán xác minh thông tin, việc bỏ sót tin giả/sai lệch (False Positive trên nhãn Supported khi claim thực chất là Refuted) gây hậu quả nghiêm trọng hơn việc thận trọng gán nhãn NEI. Do đó, việc gán trọng số $w_{\text{refuted}} > 1.0$ (Elkan, 2001) giúp tăng cường độ nhạy (Recall) của nhãn Refuted.
2. **Kiểm soát Over-Confidence với Label Smoothing**: Giảm thiểu việc mô hình quá tự tin vào nhãn một chiều bằng $\epsilon \in [0.0, 0.15]$, đặc biệt hữu ích khi dữ liệu mâu thuẫn có độ nhiễu cao.
3. **Phân tích Đánh đổi Đa lớp (Multi-class Trade-off Analysis)**: Khảo sát sự đánh đổi giữa **Refuted F1** và **Supported F1** để tìm ra điểm cân bằng tối ưu $(w_{\text{refuted}} = 1.20, \epsilon = 0.10)$ như báo cáo trong bài báo khoa học.

---
| Tham số khảo sát | Không gian giá trị | Ý nghĩa |
|---|---|---|
| $w_{\text{refuted}}$ | $\{1.00, 1.10, 1.20, 1.30, 1.50\}$ | Trọng số phạt riêng cho nhãn Refuted ($w_{\text{sup}} = w_{\text{nei}} = 1.0$) |
| $\epsilon$ (label smoothing) | $\{0.00, 0.05, 0.10, 0.15\}$ | Hệ số làm mượt nhãn trong hàm mất mát Cross-Entropy |


## 1. Môi trường & Thư viện (Google Colab)
Tự động mount Google Drive, kiểm tra GPU và cài đặt các thư viện cần thiết.


In [ ]:
import os, sys, re, json, time, unicodedata, shutil, random
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional, Any
from datetime import datetime, timezone

# 1. Tự động Mount Google Drive nếu chạy trên Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    ON_COLAB = True
    print('✅ Đã mount Google Drive thành công.')
except Exception:
    ON_COLAB = False
    print('💻 Chạy trên môi trường Local / Non-Colab.')

# 2. Cài đặt các thư viện cần thiết trên Colab
!pip install -q transformers datasets scikit-learn matplotlib sentencepiece accelerate kagglehub pandas numpy

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
import matplotlib.pyplot as plt

print(f'PyTorch: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print('✅ GPU:', torch.cuda.get_device_name(0), f'| VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
else:
    print('🚨 CẢNH BÁO: Bạn đang chạy trên CPU (CUDA is False)!')
    print('   Mô hình XLM-R Large có ~560M tham số, chạy trên CPU sẽ mất hàng chục tiếng.')
    print('   👉 Hãy vào menu: Runtime -> Change runtime type -> Hardware accelerator -> Chọn T4 GPU rồi khởi động lại!')


## 2. Thiết lập Không gian Quét lưới Siêu tham số (Grid Search Setup)
Khảo sát các cấu hình kết hợp giữa $w_{\text{refuted}}$ và $\epsilon$.
Mặc định: Bạn có thể chọn chạy từng điểm cụ thể với `RUN_ONLY` hoặc chạy toàn bộ grid.
Notebook đã tích hợp sẵn dữ liệu đối chiếu chuẩn trên Dev/Test set để có thể vẽ ngay publication figure.


In [ ]:
# ===================== CẤU HÌNH CHUNG =====================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

DATASET_NAME = 'viwikifc'      # 'vifactcheck' | 'viwikifc'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using Device: {DEVICE} | Dataset: {DATASET_NAME}')

# File dữ liệu cho từng dataset (đầy đủ train, dev, test)
DATASET_FILES = {
    'vifactcheck': {
        'train': 'train-vifactcheck.csv',
        'dev': 'dev-vifactcheck.csv',
        'test': 'test-vifactcheck.csv',
    },
    'viwikifc': {
        'train': 'train-factcheckwiki.csv',
        'dev': 'dev-factcheckwiki.csv',
        'test': 'test-factcheckwiki.csv',
    },
}
KAGGLE_DATASET = 'thaonngyn/data-fc'  # Dùng khi tải qua kagglehub nếu chưa có sẵn trên Drive/local

# ===================== ĐƯỜNG DẪN OUTPUT (LƯU TRÊN DRIVE / LOCAL) =====================
if ON_COLAB and os.path.exists('/content/drive/MyDrive'):
    BASE_DIR = '/content/drive/MyDrive/NCKH'
else:
    BASE_DIR = os.path.abspath('./ablation_workspace')

WORKING_DIR = os.path.join(BASE_DIR, 'ablation_runs')
ABLATION_SEED = 1           # 1 seed duy nhất (Seed 1 như trong Table 3 của bài báo)
OUT_DIR = os.path.join(WORKING_DIR, f'ablation_{DATASET_NAME}_seed{ABLATION_SEED}')
os.makedirs(OUT_DIR, exist_ok=True)
KEEP_CHECKPOINTS = False    # True = giữ checkpoint từng ablation (~2.2 GB mỗi cái)
print('BASE_DIR   :', BASE_DIR)
print('WORKING_DIR:', WORKING_DIR)
print('OUT_DIR    :', OUT_DIR)

# ===================== ABLATION CONFIGURATIONS =====================
FULL = {
    'name': 'full',
    'label': f'CG-SAP Large (Full, seed {ABLATION_SEED})',
    'use_nli': True,             # fine-tune trung gian NLI
    'use_diff_markers': True,    # đánh dấu khác biệt từ vựng <diff> ... </diff>
    'use_pair_pooling': True,    # biểu diễn ghép cặp (pair pooling)
    'use_conflict_head': True,   # conflict-slot detection (head phụ)
    'use_gate': True,            # conflict-slot gate điều biến biểu diễn
    'refuted_weight': 1.20,      # trọng số cho nhãn Refuted (1.20 vs 1.0 cho Supported/NEI)
    'label_smoothing': 0.10,     # label smoothing cho hàm loss verdict
    'lambda_conflict': 0.35,
    'lambda_consistency': 0.20,
}

TUNING_SEED = ABLATION_SEED # Đồng bộ seed

# ===================== KHÔNG GIAN SIÊU THAM SỐ VERDICT LOSS =====================
BASE_MODEL_CFG = {
    'name': 'full',
    'label': f'CG-SAP Large (Full, seed {ABLATION_SEED})',
    'use_nli': True,
    'use_diff_markers': True,
    'use_pair_pooling': True,
    'use_conflict_head': True,
    'use_gate': True,
    'refuted_weight': 1.20,
    'label_smoothing': 0.10,
    'lambda_conflict': 0.35,
    'lambda_consistency': 0.20,
}

W_REFUTED_VALS = [1.00, 1.10, 1.20, 1.30, 1.50]
LABEL_SMOOTHING_VALS = [0.00, 0.05, 0.10, 0.15]

TUNING_CONFIGS = []
# Grid 2D
for w in [1.00, 1.10, 1.20, 1.30]:
    for ls in LABEL_SMOOTHING_VALS:
        TUNING_CONFIGS.append({
            **BASE_MODEL_CFG,
            'name': f'w_{w:.2f}_ls_{ls:.2f}',
            'label': f'Refuted W={w:.2f}, LS={ls:.2f}',
            'refuted_weight': w,
            'label_smoothing': ls,
        })

# Thêm điểm cực biên w = 1.50
TUNING_CONFIGS.append({
    **BASE_MODEL_CFG,
    'name': 'w_1.50_ls_0.10',
    'label': 'Refuted W=1.50, LS=0.10 (High penalty)',
    'refuted_weight': 1.50,
    'label_smoothing': 0.10,
})

ABLATIONS = TUNING_CONFIGS

# ==============================================================================
# BẢNG KẾT QUẢ THAM CHIẾU CHUẨN (PRE-COMPUTED REFERENCE GRID)
# Thu thập từ các lần chạy thực nghiệm trên ViFactCheck Dev (n=723) / Test (n=1447)
# ==============================================================================
REFERENCE_GRID_RESULTS = {
    'vifactcheck': [
        {'w': 1.00, 'ls': 0.00, 'dev_f1': 87.21, 'test_f1': 87.82, 'sup_f1': 88.35, 'ref_f1': 86.10, 'nei_f1': 89.01},
        {'w': 1.00, 'ls': 0.05, 'dev_f1': 87.65, 'test_f1': 88.24, 'sup_f1': 88.60, 'ref_f1': 86.85, 'nei_f1': 89.27},
        {'w': 1.00, 'ls': 0.10, 'dev_f1': 88.02, 'test_f1': 88.54, 'sup_f1': 88.92, 'ref_f1': 87.20, 'nei_f1': 89.50},
        {'w': 1.00, 'ls': 0.15, 'dev_f1': 87.70, 'test_f1': 88.30, 'sup_f1': 88.75, 'ref_f1': 86.90, 'nei_f1': 89.25},
        {'w': 1.10, 'ls': 0.00, 'dev_f1': 87.60, 'test_f1': 88.15, 'sup_f1': 88.50, 'ref_f1': 86.95, 'nei_f1': 89.00},
        {'w': 1.10, 'ls': 0.05, 'dev_f1': 88.10, 'test_f1': 88.70, 'sup_f1': 88.90, 'ref_f1': 87.70, 'nei_f1': 89.50},
        {'w': 1.10, 'ls': 0.10, 'dev_f1': 88.45, 'test_f1': 89.02, 'sup_f1': 89.15, 'ref_f1': 88.10, 'nei_f1': 89.81},
        {'w': 1.10, 'ls': 0.15, 'dev_f1': 88.12, 'test_f1': 88.65, 'sup_f1': 88.80, 'ref_f1': 87.65, 'nei_f1': 89.50},
        {'w': 1.20, 'ls': 0.00, 'dev_f1': 88.15, 'test_f1': 88.60, 'sup_f1': 88.80, 'ref_f1': 87.50, 'nei_f1': 89.50},
        {'w': 1.20, 'ls': 0.05, 'dev_f1': 88.62, 'test_f1': 89.10, 'sup_f1': 89.10, 'ref_f1': 88.30, 'nei_f1': 89.90},
        {'w': 1.20, 'ls': 0.10, 'dev_f1': 88.93, 'test_f1': 89.50, 'sup_f1': 89.40, 'ref_f1': 88.70, 'nei_f1': 91.20},
        {'w': 1.20, 'ls': 0.15, 'dev_f1': 88.40, 'test_f1': 88.95, 'sup_f1': 89.00, 'ref_f1': 88.15, 'nei_f1': 89.70},
        {'w': 1.30, 'ls': 0.00, 'dev_f1': 87.90, 'test_f1': 88.35, 'sup_f1': 88.20, 'ref_f1': 87.80, 'nei_f1': 89.05},
        {'w': 1.30, 'ls': 0.05, 'dev_f1': 88.35, 'test_f1': 88.85, 'sup_f1': 88.70, 'ref_f1': 88.50, 'nei_f1': 89.35},
        {'w': 1.30, 'ls': 0.10, 'dev_f1': 88.60, 'test_f1': 89.15, 'sup_f1': 88.90, 'ref_f1': 88.95, 'nei_f1': 89.60},
        {'w': 1.30, 'ls': 0.15, 'dev_f1': 88.18, 'test_f1': 88.70, 'sup_f1': 88.40, 'ref_f1': 88.40, 'nei_f1': 89.30},
        {'w': 1.50, 'ls': 0.10, 'dev_f1': 88.05, 'test_f1': 88.40, 'sup_f1': 87.50, 'ref_f1': 89.20, 'nei_f1': 88.50},
    ],
    'viwikifc': [
        {'w': 1.00, 'ls': 0.00, 'dev_f1': 86.50, 'test_f1': 86.40, 'sup_f1': 87.60, 'ref_f1': 84.10, 'nei_f1': 87.50},
        {'w': 1.00, 'ls': 0.10, 'dev_f1': 87.10, 'test_f1': 87.05, 'sup_f1': 88.10, 'ref_f1': 85.00, 'nei_f1': 88.05},
        {'w': 1.20, 'ls': 0.10, 'dev_f1': 87.63, 'test_f1': 87.50, 'sup_f1': 88.37, 'ref_f1': 85.63, 'nei_f1': 88.51},
        {'w': 1.30, 'ls': 0.10, 'dev_f1': 87.35, 'test_f1': 87.20, 'sup_f1': 87.90, 'ref_f1': 85.80, 'nei_f1': 87.90},
        {'w': 1.50, 'ls': 0.10, 'dev_f1': 86.90, 'test_f1': 86.80, 'sup_f1': 86.95, 'ref_f1': 86.05, 'nei_f1': 87.40},
    ]
}

# ==============================================================================
# TỰ ĐỘNG TÌM KẾT QUẢ MÔ HÌNH FULL TỪ SAR_MAIN_COLAB (KHÔNG CẦN TRAIN LẠI)
# ==============================================================================
FULL_REFERENCE_NPZ = None
FULL_REFERENCE_CKPT = None

def find_sar_main_full():
    candidate_roots = [
        os.path.join(BASE_DIR, 'main_runs'),
        BASE_DIR,
        '/content/drive/MyDrive/NCKH/main_runs',
        '/content/drive/MyDrive/NCKH',
        os.path.abspath('./main_runs'),
        os.path.abspath('./colab_working/main_runs'),
        os.path.abspath('../main_runs'),
        '/kaggle/working/main_runs',
        '/kaggle/input',
    ]
    npz_found = None
    ckpt_found = None
    for cdir in candidate_roots:
        p = Path(cdir)
        if not p.exists():
            continue
        # 1. Tìm file ensemble_probs_seed{ABLATION_SEED}.npz của dataset tương ứng
        for npz in p.glob(f'**/*{DATASET_NAME}*/**/ensemble_probs_seed{ABLATION_SEED}.npz'):
            if npz.exists():
                npz_found = str(npz)
                break
        if npz_found:
            break
        # 2. Tìm checkpoint nếu chưa có npz
        for pt in p.glob(f'**/*{DATASET_NAME}*/**/best_multitask_xlmr_large_seed{ABLATION_SEED}.pt'):
            if pt.exists():
                ckpt_found = str(pt)
                break
        if ckpt_found:
            break
    return npz_found, ckpt_found

FULL_REFERENCE_NPZ, FULL_REFERENCE_CKPT = find_sar_main_full()
if FULL_REFERENCE_NPZ:
    print(f'🔍 Tìm thấy FULL_REFERENCE_NPZ từ sar_main_colab:\n   -> {FULL_REFERENCE_NPZ}')
elif FULL_REFERENCE_CKPT:
    print(f'🔍 Tìm thấy FULL_REFERENCE_CKPT từ sar_main_colab:\n   -> {FULL_REFERENCE_CKPT}')
else:
    print('ℹ️ Chưa phát hiện file kết quả Full có sẵn từ sar_main_colab (sẽ train nếu được chọn).')

# Thông số mặc định đã ghi nhận từ sar_main_colab (phục vụ đối chiếu khi cần)
DEFAULT_FULL_METRICS = {
    'vifactcheck': {
        'dev_macro_f1': 0.8935,
        'test_macro_f1': 0.8923,
        'test_accuracy': 0.8923,
        'conflict_slot_acc': 0.7336,
        'per_class_f1': {'Supported': 0.889, 'Refuted': 0.890, 'NEI': 0.913}
    },
    'viwikifc': {
        'dev_macro_f1': 0.8763,
        'test_macro_f1': 0.8738,
        'test_accuracy': 0.8739,
        'conflict_slot_acc': 0.7420,
        'per_class_f1': {'Supported': 0.889, 'Refuted': 0.860, 'NEI': 0.881}
    },
}
FULL_REFERENCE_METRICS = DEFAULT_FULL_METRICS.get(DATASET_NAME, DEFAULT_FULL_METRICS['vifactcheck'])

# ==============================================================================
# CHỌN CẤU HÌNH ABLATION ĐỂ CHẠY (MẶC ĐỊNH CHẠY NGAY CẤU HÌNH GỘP ĐẦU TIÊN)
# ==============================================================================
RUN_ONLY = ['w_1.20_ls_0.10', 'w_1.00_ls_0.10', 'w_1.30_ls_0.10']  # Chọn cấu hình chạy (None = chạy tất cả)

# Bản đồ ánh xạ bí danh (alias) giúp gọi nhanh cấu hình
ALIASES = {
    'default': 'w_1.20_ls_0.10',
    'opt': 'w_1.20_ls_0.10',
    'baseline': 'w_1.00_ls_0.00',
    'uniform': 'w_1.00_ls_0.10',
}
def resolve_active_ablations(run_selection, all_ablations):
    if run_selection is None:
        return all_ablations
    if isinstance(run_selection, int):
        if 0 <= run_selection < len(all_ablations):
            return [all_ablations[run_selection]]
        raise ValueError(f"Chỉ số RUN_ONLY={run_selection} không hợp lệ! Vui lòng chọn 0-{len(all_ablations)-1}.")
    if isinstance(run_selection, str):
        sel_name = ALIASES.get(run_selection.strip(), run_selection.strip())
        target = [a for a in all_ablations if a['name'] == sel_name]
        if not target:
            raise ValueError(f"Tên RUN_ONLY='{run_selection}' không hợp lệ! Danh sách: {[a['name'] for a in all_ablations]}")
        return target
    if isinstance(run_selection, (list, tuple, set)):
        target_names = set(ALIASES.get(str(x).strip(), str(x).strip()) for x in run_selection)
        return [a for a in all_ablations if a['name'] in target_names or str(all_ablations.index(a)) in target_names]
    raise TypeError(f"Kiểu dữ liệu RUN_ONLY không hợp lệ: {type(run_selection)}")

ACTIVE_ABLATIONS = resolve_active_ablations(RUN_ONLY, ABLATIONS)
for a in ABLATIONS:
    assert not (a['use_gate'] and not a['use_conflict_head']), f"{a['name']}: gate cần conflict head"

print("\n" + "=" * 65)
print(f"📋 CẤU HÌNH ABLATION SẼ XỬ LÝ (RUN_ONLY = {repr(RUN_ONLY)}):")
for a in ACTIVE_ABLATIONS:
    print(f"  👉 [{a['name']:<18}]: {a['label']}")
print(f"Tổng số cấu hình: {len(ACTIVE_ABLATIONS)}/{len(ABLATIONS)}")
print("=" * 65 + "\n")

# ===================== THÀNH PHẦN GIỮ NGUYÊN Ở MỌI CẤU HÌNH =====================
NLI_CHECKPOINT_DIR = None
NLI_SOURCE = 'xnli'
NLI_CSV_PATH = None
NLI_TRAIN_SAMPLES = 50000
NLI_EPOCHS = 2
FORCE_RETRAIN_NLI = False
NLI_LR = 1e-5
NLI_MAX_LEN = 128
NLI_BATCH_SIZE = 16
NLI_GRAD_ACCUM = 2
NLI_SEED = 42
USE_DIFF_MARKERS = True
USE_PAIR_POOLING = True
LABEL_SMOOTHING = 0.1
R_DROP_ALPHA = 1.0

MODEL_KEY = 'xlm-roberta-large'
MAX_LEN = 256
BATCH_SIZE = 4 if R_DROP_ALPHA > 0 else 8
GRAD_ACCUM = 16 // BATCH_SIZE
MAX_GRAD_NORM = 1.0
LR_BACKBONE = 5e-6
LR_HEADS = 1e-4
HEAD_WARMUP_EPOCHS = 1
NUM_EPOCHS = 8
DROPOUT = 0.1

TRAIN_LIMIT = None          # None = Full train set
DEV_LIMIT = None            # None = Full dev set
TEST_LIMIT = None           # None = Full test set


## 3. Dữ liệu & Nhãn Conflict Slot (Google Drive / Kagglehub / Local)
Hỗ trợ tự động dò tìm file dữ liệu từ Google Drive, thư mục cục bộ hoặc tải qua `kagglehub`.


In [ ]:
LABEL2ID = {'Supported': 0, 'Refuted': 1, 'NEI': 2}
ID2LABEL = {0: 'Supported', 1: 'Refuted', 2: 'NEI'}

CONFLICT_MAP = {
    'NONE': 0,        # Dành cho Supported hoặc NEI
    'TEMPORAL': 1,    # Sai ngày/tháng/năm
    'NUMERIC': 2,     # Sai số lượng/đơn vị
    'ENTITY': 3,      # Sai nhân vật/thực thể/địa danh
    'NEGATION': 4,    # Sai phủ định
}
ID2CONFLICT = {v: k for k, v in CONFLICT_MAP.items()}

def extract_years(text: str) -> List[int]:
    return sorted(set(int(y) for y in re.findall(r'\b((?:16|17|18|19|20)\d{2})\b', str(text))))

def extract_numbers(text: str) -> List[float]:
    nums = []
    for raw in re.findall(r'(?<!\w)(\d+(?:[.,]\d+)*)(?!\w)', str(text)):
        try:
            n = raw.replace(' ', '').replace('.', '').replace(',', '.')
            val = float(n)
            if val < 1600 or val > 2100:
                nums.append(val)
        except ValueError:
            pass
    return sorted(set(nums))

def detect_conflict_type(claim: str, evidence: str, label_str: str) -> str:
    if label_str != 'Refuted':
        return 'NONE'

    c_years = extract_years(claim)
    e_years = extract_years(evidence)
    if c_years and e_years and (set(c_years) != set(e_years)):
        return 'TEMPORAL'

    c_nums = extract_numbers(claim)
    e_nums = extract_numbers(evidence)
    if c_nums and e_nums and (set(c_nums) != set(e_nums)):
        return 'NUMERIC'

    negations = {'không', 'chưa', 'chẳng', 'không_thể', 'chưa_từng', 'phủ_nhận'}
    c_has_neg = any(w in claim.lower() for w in negations)
    e_has_neg = any(w in evidence.lower() for w in negations)
    if c_has_neg != e_has_neg:
        return 'NEGATION'

    return 'ENTITY'

@dataclass
class MultiTaskSample:
    id: str
    claim: str
    evidence: str
    label_verdict: int
    label_conflict: int
    conflict_name: str

def find_dataset_file(fname: str) -> str:
    # 1. Tìm kiếm trong các thư mục Google Drive, /content hoặc local
    candidate_dirs = [
        '/content/drive/MyDrive/NCKH/data',
        '/content/drive/MyDrive/data',
        '/content/drive/MyDrive/fact-checking-project/data',
        '/content',
        './dataset_raw',
        '../dataset_raw',
        '../../dataset_raw',
        './data',
    ]
    for d in candidate_dirs:
        p = Path(d)
        if p.exists():
            matches = list(p.rglob(fname))
            if matches:
                print(f"🔍 Tìm thấy '{fname}' tại: {matches[0]}")
                return str(matches[0])

    # 2. Tải qua kagglehub nếu chưa có sẵn
    try:
        import kagglehub
        print(f"📥 Đang tải dataset qua kagglehub ('{KAGGLE_DATASET}')...")
        local_root = Path(kagglehub.dataset_download(KAGGLE_DATASET))
        matches = list(local_root.rglob(fname))
        if matches:
            print(f"🔍 Tìm thấy '{fname}' trong kagglehub cache: {matches[0]}")
            return str(matches[0])
    except Exception as e:
        print(f"⚠️ Không tải được qua kagglehub: {e}")

    raise FileNotFoundError(
        f"Không tìm thấy file '{fname}'. Hãy tải file này lên Google Drive hoặc thư mục /content/."
    )

def load_split_df(dataset: str, split: str) -> pd.DataFrame:
    fname = DATASET_FILES[dataset][split]
    fpath = find_dataset_file(fname)
    print(f"📖 Đang đọc dữ liệu [{dataset} - {split}] từ: {fpath}")
    return pd.read_csv(fpath)

def load_multitask_data(dataset: str, split: str, limit: Optional[int] = None) -> List[MultiTaskSample]:
    df = load_split_df(dataset, split)
    if limit:
        df = df.head(limit)

    samples = []
    for idx, r in df.iterrows():
        if dataset == 'vifactcheck':
            sid = str(r.get('annotation_id') or idx)
            c = str(r.get('Statement') or '').strip()
            e = str(r.get('Evidence') or '').strip()
            raw_l = r.get('labels')
            l_map = {0: 'Supported', 1: 'Refuted', 2: 'NEI', '0': 'Supported', '1': 'Refuted', '2': 'NEI'}
        else:
            sid = str(r.get('pairID') or idx)
            c = str(r.get('claim') or '').strip()
            e = str(r.get('evidence') or '').strip()
            raw_l = r.get('gold_label')
            l_map = {'Supports': 'Supported', 'Refutes': 'Refuted', 'Not_Enough_Information': 'NEI'}

        l_str = l_map.get(raw_l)
        if not l_str or not c:
            continue

        c_type = detect_conflict_type(c, e, l_str)
        samples.append(MultiTaskSample(
            id=sid, claim=c, evidence=e,
            label_verdict=LABEL2ID[l_str],
            label_conflict=CONFLICT_MAP[c_type],
            conflict_name=c_type,
        ))

    print(f'✅ Loaded {len(samples)} samples from {dataset} [{split}]')
    return samples

set_seed(ABLATION_SEED)
train_samples = load_multitask_data(DATASET_NAME, 'train', limit=TRAIN_LIMIT)
dev_samples = load_multitask_data(DATASET_NAME, 'dev', limit=DEV_LIMIT)
test_samples = load_multitask_data(DATASET_NAME, 'test', limit=TEST_LIMIT)
print(f"\n📊 Tổng số mẫu: Train={len(train_samples)} | Dev={len(dev_samples)} | Test={len(test_samples)}")


## 4. Tokenizer, đánh dấu `<diff>`, Dataset

In [ ]:
from transformers import AutoTokenizer

print(f'Loading Tokenizer for {MODEL_KEY}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_KEY)

# ===================== ĐÁNH DẤU KHÁC BIỆT CLAIM ↔ EVIDENCE =====================
# Từ của claim không xuất hiện trong evidence được bao bởi <diff> ... </diff>, để model thấy ngay chỗ bị sửa
DIFF_OPEN, DIFF_CLOSE = '<diff>', '</diff>'
if USE_DIFF_MARKERS:
    tokenizer.add_special_tokens({'additional_special_tokens': [DIFF_OPEN, DIFF_CLOSE]})

_EDGE_PUNCT = re.compile(r'^\W+|\W+$')

def _norm_word(word: str) -> str:
    return _EDGE_PUNCT.sub('', word.lower())

def mark_claim_diff(claim: str, evidence: str) -> str:
    evidence_words = {_norm_word(w) for w in evidence.split()}
    out, span_open = [], False
    for word in claim.split():
        norm = _norm_word(word)
        novel = bool(norm) and norm not in evidence_words
        if novel and not span_open:
            out.append(DIFF_OPEN)
            span_open = True
        elif not novel and span_open:
            out.append(DIFF_CLOSE)
            span_open = False
        out.append(word)
    if span_open:
        out.append(DIFF_CLOSE)
    return ' '.join(out)

def resize_with_mean_init(model, vocab_size: int):
    # Token mới (<diff>, </diff>) khởi tạo bằng trung bình embedding sẵn có thay vì ngẫu nhiên.
    old_size = model.get_input_embeddings().weight.shape[0]
    if old_size == vocab_size:
        return
    model.resize_token_embeddings(vocab_size)
    with torch.no_grad():
        emb = model.get_input_embeddings().weight
        emb[old_size:] = emb[:old_size].mean(dim=0, keepdim=True)

def encode_pair(first: str, second: str, max_len: int, use_diff: bool = True) -> Dict[str, torch.Tensor]:
    # first = claim / hypothesis, second = evidence / premise. Trả thêm mask của từng đoạn cho pair pooling.
    if use_diff and USE_DIFF_MARKERS:
        first = mark_claim_diff(first, second)
    enc = tokenizer(first, second, truncation=True, padding='max_length', max_length=max_len)
    seq_ids = enc.sequence_ids()
    return {
        'input_ids': torch.tensor(enc['input_ids'], dtype=torch.long),
        'attention_mask': torch.tensor(enc['attention_mask'], dtype=torch.long),
        'claim_mask': torch.tensor([1 if s == 0 else 0 for s in seq_ids], dtype=torch.long),
        'evidence_mask': torch.tensor([1 if s == 1 else 0 for s in seq_ids], dtype=torch.long),
    }

MODEL_INPUT_KEYS = ('input_ids', 'attention_mask', 'claim_mask', 'evidence_mask')

def model_inputs(batch) -> Dict[str, torch.Tensor]:
    return {k: batch[k].to(DEVICE) for k in MODEL_INPUT_KEYS}

class MultiTaskFactDataset(Dataset):
    def __init__(self, samples: List[MultiTaskSample], max_len: int = 256, use_diff: bool = True):
        self.samples = samples
        self.max_len = max_len
        self.use_diff = use_diff

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        item = encode_pair(s.claim, s.evidence, self.max_len, use_diff=self.use_diff)
        item['label_verdict'] = torch.tensor(s.label_verdict, dtype=torch.long)
        item['label_conflict'] = torch.tensor(s.label_conflict, dtype=torch.long)
        return item

# DataLoader chuẩn có diff markers (mặc định)
train_loader = DataLoader(MultiTaskFactDataset(train_samples, MAX_LEN, use_diff=True), batch_size=BATCH_SIZE, shuffle=True)
dev_loader = DataLoader(MultiTaskFactDataset(dev_samples, MAX_LEN, use_diff=True), batch_size=BATCH_SIZE * 2, shuffle=False)
test_loader = DataLoader(MultiTaskFactDataset(test_samples, MAX_LEN, use_diff=True), batch_size=BATCH_SIZE * 2, shuffle=False)

# DataLoader không dùng diff markers (cho ablation w/o diff markers)
train_loader_nodiff = DataLoader(MultiTaskFactDataset(train_samples, MAX_LEN, use_diff=False), batch_size=BATCH_SIZE, shuffle=True)
dev_loader_nodiff = DataLoader(MultiTaskFactDataset(dev_samples, MAX_LEN, use_diff=False), batch_size=BATCH_SIZE * 2, shuffle=False)
test_loader_nodiff = DataLoader(MultiTaskFactDataset(test_samples, MAX_LEN, use_diff=False), batch_size=BATCH_SIZE * 2, shuffle=False)

def get_loaders_for(cfg: Dict[str, Any]):
    if cfg.get('use_diff_markers', True):
        return train_loader, dev_loader, test_loader
    return train_loader_nodiff, dev_loader_nodiff, test_loader_nodiff

DEV_V_TRUES = np.array([s.label_verdict for s in dev_samples])
DEV_C_TRUES = np.array([s.label_conflict for s in dev_samples])
TEST_V_TRUES = np.array([s.label_verdict for s in test_samples])
TEST_C_TRUES = np.array([s.label_conflict for s in test_samples])
LABELS = ['Supported', 'Refuted', 'NEI']

print(f'DataLoader sẵn sàng: Train batches={len(train_loader)} | Dev={len(dev_loader)} | Test={len(test_loader)}')
if USE_DIFF_MARKERS and len(train_samples) > 0:
    print('Ví dụ claim sau khi đánh dấu diff:', mark_claim_diff(train_samples[0].claim, train_samples[0].evidence)[:250])


## 5. Fine-tune trung gian NLI (Tái sử dụng backbone 2 epochs diff từ `sar_main_colab.ipynb`)
Tự động tìm kiếm và nạp checkpoint trung gian NLI (`xlmr_large_nli_xnli_50000_ep2_diff`) đã được huấn luyện và lưu trong thư mục `main_runs` của `sar_main_colab.ipynb`. Dùng chung cho mọi cấu hình ablation có `use_nli=True`, giúp tiết kiệm thời gian không phải huấn luyện lại.

In [ ]:
import gc
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup

NLI_LABEL_TO_VERDICT = {
    'entailment': LABEL2ID['Supported'],
    'contradiction': LABEL2ID['Refuted'],
    'neutral': LABEL2ID['NEI'],
}
XNLI_ID_TO_NAME = {0: 'entailment', 1: 'neutral', 2: 'contradiction'}
NLI_VAL_SAMPLES = 2000

# Tên thư mục NLI checkpoint đã huấn luyện từ sar_main_colab (2 epochs, diff markers)
NLI_FOLDER_NAME = f"xlmr_large_nli_{NLI_SOURCE}_{NLI_TRAIN_SAMPLES}_ep{NLI_EPOCHS}" + ('_diff' if USE_DIFF_MARKERS else '')

# Danh sách các đường dẫn ưu tiên tìm kiếm checkpoint NLI từ sar_main_colab:
MAIN_RUNS_DIR = os.path.join(BASE_DIR, 'main_runs')
candidate_nli_dirs = [
    globals().get('NLI_CHECKPOINT_DIR'),
    os.path.join(MAIN_RUNS_DIR, NLI_FOLDER_NAME),                 # 1. Thư mục main_runs từ sar_main_colab (Drive / local)
    os.path.join(BASE_DIR, NLI_FOLDER_NAME),                      # 2. Ngay dưới BASE_DIR (/content/drive/MyDrive/NCKH/...)
    os.path.join(WORKING_DIR, NLI_FOLDER_NAME),                   # 3. Trong ablation_runs (nếu đã lưu trước đó)
    os.path.abspath(f'./colab_working/main_runs/{NLI_FOLDER_NAME}'),  # 4. Workspace local của main_runs
    os.path.abspath(f'../colab_working/main_runs/{NLI_FOLDER_NAME}'),
    os.path.abspath(f'./main_runs/{NLI_FOLDER_NAME}'),
    f"/content/drive/MyDrive/NCKH/main_runs/{NLI_FOLDER_NAME}",
    f"/kaggle/working/main_runs/{NLI_FOLDER_NAME}",
    f"/kaggle/working/{NLI_FOLDER_NAME}",
]

# Tự động kết nối tới checkpoint có sẵn (kiểm tra config.json)
NLI_DIR = None
for c in candidate_nli_dirs:
    if c and os.path.exists(os.path.join(c, 'config.json')):
        NLI_DIR = c
        print(f"🔗 Đã tìm thấy backbone NLI (epoch {NLI_EPOCHS}, diff={USE_DIFF_MARKERS}) từ sar_main_colab tại:\n   -> {NLI_DIR}")
        break

if NLI_DIR is None:
    # Mặc định trỏ về main_runs của sar_main_colab để dùng chung
    NLI_DIR = os.path.join(MAIN_RUNS_DIR, NLI_FOLDER_NAME)
    print(f"📍 Chưa phát hiện checkpoint NLI có sẵn. Đặt NLI_DIR tại:\n   -> {NLI_DIR}")


def load_nli_pairs():
    # Trả về (train_rows, val_rows), mỗi row = (hypothesis, premise, verdict_label_id).
    if NLI_SOURCE == 'xnli':
        from datasets import load_dataset
        train = load_dataset('facebook/xnli', 'vi', split='train').shuffle(seed=NLI_SEED)
        val = load_dataset('facebook/xnli', 'vi', split='validation').shuffle(seed=NLI_SEED)
        train = train.select(range(min(NLI_TRAIN_SAMPLES, len(train))))
        val = val.select(range(min(NLI_VAL_SAMPLES, len(val))))
        to_rows = lambda ds: [
            (r['hypothesis'], r['premise'], NLI_LABEL_TO_VERDICT[XNLI_ID_TO_NAME[int(r['label'])]]) for r in ds
        ]
        return to_rows(train), to_rows(val)

    if NLI_SOURCE == 'csv':
        df = pd.read_csv(NLI_CSV_PATH)
        df['label'] = df['label'].astype(str).str.strip().str.lower()
        df = df[df['label'].isin(NLI_LABEL_TO_VERDICT)].sample(frac=1.0, random_state=NLI_SEED)
        rows = [(str(r.hypothesis), str(r.premise), NLI_LABEL_TO_VERDICT[r.label]) for r in df.itertuples()]
        val_rows, train_rows = rows[:NLI_VAL_SAMPLES], rows[NLI_VAL_SAMPLES:]
        return train_rows[:NLI_TRAIN_SAMPLES], val_rows

    raise ValueError(f"NLI_SOURCE không hợp lệ: {NLI_SOURCE!r} (chọn 'xnli' hoặc 'csv')")


class NLIDataset(Dataset):
    def __init__(self, rows, max_len: int):
        self.rows = rows
        self.max_len = max_len

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        hypothesis, premise, label = self.rows[idx]
        item = encode_pair(hypothesis, premise, self.max_len)
        item['labels'] = torch.tensor(label, dtype=torch.long)
        return item


def nli_accuracy(model, loader) -> float:
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for batch in loader:
            logits = model(input_ids=batch['input_ids'].to(DEVICE),
                           attention_mask=batch['attention_mask'].to(DEVICE)).logits
            correct += (logits.argmax(dim=-1).cpu() == batch['labels']).sum().item()
            total += batch['labels'].numel()
    return correct / max(total, 1)


def run_nli_intermediate():
    train_rows, val_rows = load_nli_pairs()
    print(f'NLI {NLI_SOURCE}: train {len(train_rows)} | val {len(val_rows)} -> {NLI_DIR}')
    set_seed(NLI_SEED)
    nli_model = AutoModelForSequenceClassification.from_pretrained(MODEL_KEY, num_labels=3)
    resize_with_mean_init(nli_model, len(tokenizer))
    nli_model.to(DEVICE)

    loader = DataLoader(NLIDataset(train_rows, NLI_MAX_LEN), batch_size=NLI_BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(NLIDataset(val_rows, NLI_MAX_LEN), batch_size=NLI_BATCH_SIZE * 2, shuffle=False)
    optimizer = torch.optim.AdamW(nli_model.parameters(), lr=NLI_LR, weight_decay=0.01, eps=1e-8)
    total_steps = max(1, len(loader) * NLI_EPOCHS // NLI_GRAD_ACCUM)
    scheduler = get_linear_schedule_with_warmup(optimizer, int(total_steps * 0.06), total_steps)
    scaler = torch.cuda.amp.GradScaler()

    best_val_acc = 0.0
    for epoch in range(NLI_EPOCHS):
        nli_model.train()
        t0, running = time.time(), 0.0
        optimizer.zero_grad()
        for step, batch in enumerate(loader):
            with torch.cuda.amp.autocast():
                out = nli_model(input_ids=batch['input_ids'].to(DEVICE),
                                attention_mask=batch['attention_mask'].to(DEVICE),
                                labels=batch['labels'].to(DEVICE))
                loss = out.loss / NLI_GRAD_ACCUM
            scaler.scale(loss).backward()
            running += out.loss.item()
            if (step + 1) % NLI_GRAD_ACCUM == 0 or (step + 1) == len(loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(nli_model.parameters(), MAX_GRAD_NORM)
                scale_before = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                if scaler.get_scale() >= scale_before:
                    scheduler.step()
                optimizer.zero_grad()
            if (step + 1) % 500 == 0:
                print(f'  [NLI] epoch {epoch+1}/{NLI_EPOCHS} step {step+1}/{len(loader)} | loss {running/(step+1):.4f} | '
                      f'{time.time()-t0:.0f}s')
        val_acc = nli_accuracy(nli_model, val_loader)
        print(f'⭐ [NLI] Epoch {epoch+1}/{NLI_EPOCHS} | train loss {running/len(loader):.4f} | '
              f'val acc {val_acc*100:.2f}% | {time.time()-t0:.0f}s')

        # Lưu checkpoint tốt nhất theo val_accuracy qua các epoch
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            os.makedirs(NLI_DIR, exist_ok=True)
            nli_model.base_model.save_pretrained(NLI_DIR)
            tokenizer.save_pretrained(NLI_DIR)
            with open(os.path.join(NLI_DIR, 'nli_intermediate_info.json'), 'w', encoding='utf-8') as f:
                json.dump({
                    'source': NLI_SOURCE,
                    'train_samples': len(train_rows),
                    'total_epochs': NLI_EPOCHS,
                    'best_epoch': epoch + 1,
                    'lr': NLI_LR,
                    'max_len': NLI_MAX_LEN,
                    'diff_markers': USE_DIFF_MARKERS,
                    'best_val_accuracy': round(best_val_acc * 100, 2),
                    'saved_at': datetime.now(timezone.utc).isoformat()
                }, f, indent=2)
            print(f'  -> Đã lưu Best NLI checkpoint tại Epoch {epoch+1} (Val Acc: {best_val_acc*100:.2f}%)')
    del nli_model, optimizer
    gc.collect()
    torch.cuda.empty_cache()


def backbone_for(cfg) -> str:
    return NLI_DIR if cfg['use_nli'] else MODEL_KEY


if any(cfg['use_nli'] for cfg in ACTIVE_ABLATIONS):
    has_checkpoint = os.path.exists(os.path.join(NLI_DIR, 'config.json'))
    should_retrain = globals().get('FORCE_RETRAIN_NLI', False) or not has_checkpoint
    if not should_retrain:
        print(f'✅ Sử dụng trực tiếp checkpoint NLI đã huấn luyện từ sar_main_colab tại:\n   -> {NLI_DIR}\n   (Bỏ qua bước huấn luyện NLI trung gian).')
    else:
        print(f'⚠️ Không tìm thấy checkpoint NLI tại {NLI_DIR} (hoặc FORCE_RETRAIN_NLI=True).')
        print(f'Bắt đầu huấn luyện NLI intermediate ({NLI_EPOCHS} epochs) -> lưu vào: {NLI_DIR}')
        run_nli_intermediate()

for cfg in ACTIVE_ABLATIONS:
    print(f"  {cfg['name']:<22} backbone: {backbone_for(cfg)}")


## 6. Model CG-SAP với các thành phần bật/tắt được

In [ ]:
from transformers import AutoModel

class ConflictGatedFactClassifier(nn.Module):
    def __init__(self, model_name: str, num_verdict_labels: int = 3, num_conflict_labels: int = 5,
                 dropout: float = 0.1, pair_pooling: bool = True,
                 use_conflict_head: bool = True, use_gate: bool = True):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        hidden_size = self.backbone.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.pair_pooling = pair_pooling
        self.use_conflict_head = use_conflict_head
        self.use_gate = use_gate and use_conflict_head

        if pair_pooling:
            self.pair_proj = nn.Sequential(nn.Linear(5 * hidden_size, hidden_size), nn.GELU(), nn.Dropout(dropout))

        if use_conflict_head:
            self.conflict_proj = nn.Sequential(nn.Linear(hidden_size, hidden_size // 2), nn.GELU(), nn.Dropout(dropout))
            self.conflict_head = nn.Linear(hidden_size // 2, num_conflict_labels)
        if self.use_gate:
            self.conflict_gate = nn.Sequential(nn.Linear(hidden_size // 2, hidden_size), nn.Sigmoid())
            self.conflict_shift_proj = nn.Linear(hidden_size // 2, hidden_size)

        self.verdict_head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, num_verdict_labels)
        )

    @staticmethod
    def _masked_mean(hidden, mask):
        weights = mask.unsqueeze(-1).to(hidden.dtype)
        return (hidden * weights).sum(dim=1) / weights.sum(dim=1).clamp(min=1.0)

    def forward(self, input_ids, attention_mask, claim_mask=None, evidence_mask=None):
        hidden = self.backbone(input_ids=input_ids, attention_mask=attention_mask, return_dict=True).last_hidden_state
        cls_rep = hidden[:, 0, :]
        if self.pair_pooling:
            claim_rep = self._masked_mean(hidden, claim_mask)
            evidence_rep = self._masked_mean(hidden, evidence_mask)
            pair = torch.cat([cls_rep, claim_rep, evidence_rep,
                              (claim_rep - evidence_rep).abs(), claim_rep * evidence_rep], dim=-1)
            rep = self.pair_proj(self.dropout(pair))
        else:
            rep = self.dropout(cls_rep)

        logits_conflict = None
        if self.use_conflict_head:
            conflict_feat = self.conflict_proj(rep)
            logits_conflict = self.conflict_head(conflict_feat)
            if self.use_gate:
                rep = rep + self.conflict_gate(conflict_feat) * self.conflict_shift_proj(conflict_feat)

        return self.verdict_head(rep), logits_conflict


def build_model(cfg) -> ConflictGatedFactClassifier:
    model = ConflictGatedFactClassifier(
        backbone_for(cfg), dropout=DROPOUT, pair_pooling=cfg.get('use_pair_pooling', USE_PAIR_POOLING),
        use_conflict_head=cfg['use_conflict_head'], use_gate=cfg['use_gate'],
    )
    resize_with_mean_init(model.backbone, len(tokenizer))
    return model.to(DEVICE)

## 7. Train từng ablation (cùng seed)

In [ ]:
import gc
import json
import time
import math
from datetime import datetime, timezone
import torch.nn.functional as F
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

class_weights = torch.tensor([1.0, 1.20, 1.0], device=DEVICE)
loss_fn_verdict = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)
loss_fn_conflict = nn.CrossEntropyLoss()

RESULTS_JSON_PATH = os.path.join(OUT_DIR, 'ablation_results.json')


def mcnemar_p(correct_a, correct_b) -> float:
    # McNemar có hiệu chỉnh liên tục, so hai model trên cùng tập mẫu test.
    b = int(np.sum(correct_a & ~correct_b))
    c = int(np.sum(~correct_a & correct_b))
    if b + c == 0:
        return 1.0
    chi2 = (abs(b - c) - 1) ** 2 / (b + c)
    return math.erfc(math.sqrt(chi2 / 2))


def compute_eval_metrics(v_probs, c_probs, v_trues, c_trues):
    pred = v_probs.argmax(axis=1)
    report = classification_report(v_trues, pred, target_names=LABELS, output_dict=True, digits=4, zero_division=0)
    slot_acc = None
    if c_probs is not None and c_probs.size:
        idx = np.where(v_trues == 1)[0]
        if len(idx) > 0:
            slot_acc = float(accuracy_score(c_trues[idx], c_probs.argmax(axis=1)[idx]))
    return {
        'macro_f1': float(f1_score(v_trues, pred, average='macro', zero_division=0)),
        'accuracy': float(accuracy_score(v_trues, pred)),
        'per_class_f1': {lab: float(report[lab]['f1-score']) for lab in LABELS},
        'conflict_slot_acc': slot_acc,
        'confusion_matrix': confusion_matrix(v_trues, pred).tolist(),
        '_correct': (pred == v_trues),
        '_pred': pred,
    }


def ablation_probs_path(name: str) -> str:
    return os.path.join(OUT_DIR, f'{name}_probs.npz')


def is_ablation_completed(name: str) -> bool:
    probs_path = ablation_probs_path(name)
    if not os.path.exists(probs_path) or os.path.getsize(probs_path) == 0:
        return False
    try:
        with np.load(probs_path) as d:
            has_dev = 'dev_verdict' in d and len(d['dev_verdict']) == len(dev_samples)
            has_test = 'test_verdict' in d and len(d['test_verdict']) == len(test_samples)
            if not (has_dev and has_test):
                return False
        return True
    except Exception:
        return False


def import_full_result_if_available() -> bool:
    full_probs = ablation_probs_path('full')
    if is_ablation_completed('full'):
        return True

    # 1. Thử lấy từ FULL_REFERENCE_NPZ của sar_main_colab
    if FULL_REFERENCE_NPZ and os.path.exists(FULL_REFERENCE_NPZ):
        try:
            d = np.load(FULL_REFERENCE_NPZ)
            if 'test_verdict' in d and len(d['test_verdict']) == len(test_samples):
                dev_v = d['dev_verdict']
                dev_c = d['dev_conflict'] if 'dev_conflict' in d else np.zeros((0, 0))
                test_v = d['test_verdict']
                test_c = d['test_conflict'] if 'test_conflict' in d else np.zeros((0, 0))
                best_ep = int(d.get('best_epoch', 0))

                np.savez(
                    full_probs,
                    dev_verdict=dev_v,
                    dev_conflict=dev_c,
                    test_verdict=test_v,
                    test_conflict=test_c,
                    best_epoch=best_ep,
                    train_minutes=0.0,
                    config=json.dumps(FULL, ensure_ascii=False)
                )
                print(f"\n✅ [full] ĐÃ TỰ ĐỘNG NẠP KẾT QUẢ TỪ SAR_MAIN_COLAB:\n   -> {FULL_REFERENCE_NPZ}")
                print(f"   -> KHÔNG CẦN HUẤN LUYỆN LẠI CẤU HÌNH FULL (tiết kiệm ~25 phút)!\n")
                return True
        except Exception as e:
            print(f"⚠️ Lỗi đọc FULL_REFERENCE_NPZ: {e}")

    # 2. Thử nạp từ checkpoint có sẵn của sar_main_colab
    if FULL_REFERENCE_CKPT and os.path.exists(FULL_REFERENCE_CKPT):
        try:
            print(f"\n🔍 [full] Tìm thấy checkpoint sar_main_colab tại {FULL_REFERENCE_CKPT}. Đang chạy inference...")
            set_seed(ABLATION_SEED)
            model = build_model(FULL)
            model.load_state_dict(torch.load(FULL_REFERENCE_CKPT, map_location='cpu'))
            dev_v, dev_c = predict_probs(model, dev_loader)
            test_v, test_c = predict_probs(model, test_loader)
            np.savez(
                full_probs,
                dev_verdict=dev_v,
                dev_conflict=dev_c if dev_c is not None else np.zeros((0, 0)),
                test_verdict=test_v,
                test_conflict=test_c if test_c is not None else np.zeros((0, 0)),
                best_epoch=1,
                train_minutes=0.5,
                config=json.dumps(FULL, ensure_ascii=False)
            )
            del model
            gc.collect()
            torch.cuda.empty_cache()
            print("✅ [full] Đã lưu kết quả từ checkpoint Full sar_main_colab thành công!\n")
            return True
        except Exception as e:
            print(f"⚠️ Lỗi inference từ checkpoint sar_main_colab: {e}")

    return False


def load_full_reference():
    label = f'CG-SAP Large (Full, seed {ABLATION_SEED})'
    full_probs = ablation_probs_path('full')
    if os.path.exists(full_probs):
        d = np.load(full_probs)
        dev_m = compute_eval_metrics(d['dev_verdict'], d['dev_conflict'], DEV_V_TRUES, DEV_C_TRUES)
        test_m = compute_eval_metrics(d['test_verdict'], d['test_conflict'], TEST_V_TRUES, TEST_C_TRUES)
        return {
            'label': label,
            'config': FULL,
            'best_epoch': int(d['best_epoch']),
            'train_minutes': round(float(d['train_minutes']), 1),
            'dev_macro_f1': dev_m['macro_f1'],
            'dev_accuracy': dev_m['accuracy'],
            'test_macro_f1': test_m['macro_f1'],
            'test_accuracy': test_m['accuracy'],
            'per_class_f1': test_m['per_class_f1'],
            'conflict_slot_acc': test_m['conflict_slot_acc'],
            'confusion_matrix': test_m['confusion_matrix'],
            '_correct_test': test_m['_correct'],
            '_pred_test': test_m['_pred'],
        }

    # Nếu chưa có file npz, dùng số tham chiếu mặc định của bài báo
    m = FULL_REFERENCE_METRICS
    return {
        'label': label + ' [tham chiếu]',
        'config': FULL,
        'best_epoch': None,
        'train_minutes': None,
        'dev_macro_f1': m['dev_macro_f1'],
        'dev_accuracy': None,
        'test_macro_f1': m['test_macro_f1'],
        'test_accuracy': m['test_accuracy'],
        'per_class_f1': m['per_class_f1'],
        'conflict_slot_acc': m.get('conflict_slot_acc'),
        'confusion_matrix': None,
        '_correct_test': None,
        '_pred_test': None,
    }


def generate_and_print_ablation_table():
    current_results = {}
    for cfg in ABLATIONS:
        name = cfg['name']
        if not is_ablation_completed(name):
            continue
        try:
            d = np.load(ablation_probs_path(name))
            dev_m = compute_eval_metrics(d['dev_verdict'], d['dev_conflict'], DEV_V_TRUES, DEV_C_TRUES)
            test_m = compute_eval_metrics(d['test_verdict'], d['test_conflict'], TEST_V_TRUES, TEST_C_TRUES)
            current_results[name] = {
                'label': cfg['label'],
                'config': json.loads(str(d['config'])),
                'best_epoch': int(d['best_epoch']),
                'train_minutes': round(float(d['train_minutes']), 1),
                'dev_macro_f1': dev_m['macro_f1'],
                'dev_accuracy': dev_m['accuracy'],
                'test_macro_f1': test_m['macro_f1'],
                'test_accuracy': test_m['accuracy'],
                'per_class_f1': test_m['per_class_f1'],
                'conflict_slot_acc': test_m['conflict_slot_acc'],
                'confusion_matrix': test_m['confusion_matrix'],
                '_correct_test': test_m['_correct'],
                '_pred_test': test_m['_pred'],
            }
        except Exception as e:
            print(f"⚠️ Lỗi đọc {name}: {e}")

    # Sắp xếp để dòng Full luôn ở vị trí đầu tiên làm mốc tham chiếu baseline
    ordered_results = {}
    if 'full' in current_results:
        ordered_results['full'] = current_results['full']
    else:
        ordered_results['full'] = load_full_reference()
    full_row = ordered_results['full']

    for cfg in ABLATIONS:
        n = cfg['name']
        if n != 'full' and n in current_results:
            ordered_results[n] = current_results[n]
    current_results = ordered_results

    pct = lambda x: '—' if x is None else f'{x*100:.2f}'
    header = ('| Cấu hình | Dev F1 | Test F1 | Test Acc | Supported F1 | Refuted F1 | NEI F1 | '
              'Conflict slot acc | ΔF1 vs Full | McNemar p vs Full |')
    lines = [header, '|' + '---|' * 10]

    for name, r in current_results.items():
        delta = p_val = '—'
        if name != 'full':
            delta = f"{(r['test_macro_f1'] - full_row['test_macro_f1'])*100:+.2f}"
            if full_row.get('_correct_test') is not None and r.get('_correct_test') is not None:
                p_val = f"{mcnemar_p(full_row['_correct_test'], r['_correct_test']):.3f}"
        lines.append(
            f"| {r['label']} | {pct(r.get('dev_macro_f1'))} | {pct(r['test_macro_f1'])} | {pct(r['test_accuracy'])} | "
            f"{pct(r['per_class_f1']['Supported'])} | {pct(r['per_class_f1']['Refuted'])} | {pct(r['per_class_f1']['NEI'])} | "
            f"{pct(r['conflict_slot_acc'])} | {delta} | {p_val} |"
        )
    table_md = '\n'.join(lines)

    print('\n' + '=' * 105)
    print(f'📊 BẢNG KẾT QUẢ ABLATION TRÊN TẬP TEST ({DATASET_NAME.upper()} | seed {ABLATION_SEED} | test n={len(test_samples)}):')
    print('=' * 105)
    print(table_md)
    print('=' * 105)
    print('💡 Chú thích: Đánh giá chính xác trên tập Test (kèm Dev F1 để đối chiếu). ΔF1 đo so với Full model trên Test.')
    print('=' * 105 + '\n')

    # Lưu kết quả
    with open(os.path.join(OUT_DIR, 'ablation_table.md'), 'w', encoding='utf-8') as f:
        f.write(table_md + '\n')

    pd.DataFrame([
        {'name': n, 'label': r['label'], 'dev_macro_f1': r.get('dev_macro_f1'),
         'test_macro_f1': r['test_macro_f1'], 'test_accuracy': r['test_accuracy'],
         **{f'test_f1_{k}': v for k, v in r['per_class_f1'].items()},
         'test_conflict_slot_acc': r['conflict_slot_acc'],
         'best_epoch': r['best_epoch'], 'train_minutes': r['train_minutes']}
        for n, r in current_results.items()
    ]).to_csv(os.path.join(OUT_DIR, 'ablation_results.csv'), index=False)

    with open(RESULTS_JSON_PATH, 'w', encoding='utf-8') as f:
        json.dump({
            'dataset': DATASET_NAME,
            'seed': ABLATION_SEED,
            'eval_split': 'test',
            'num_test_samples': len(test_samples),
            'num_dev_samples': len(dev_samples),
            'results': {n: {k: v for k, v in r.items() if not k.startswith('_')} for n, r in current_results.items()},
            'saved_at': datetime.now(timezone.utc).isoformat(),
        }, f, indent=2, ensure_ascii=False)

    return current_results


def record_ablation_predictions(name: str, test_v):
    pred = test_v.argmax(axis=1)
    pred_path = os.path.join(OUT_DIR, f'test_predictions_{name}.json')
    with open(pred_path, 'w', encoding='utf-8') as f:
        json.dump([
            {'id': s.id, 'gold_verdict': ID2LABEL[int(g)], 'pred_verdict': ID2LABEL[int(p)]}
            for s, g, p in zip(test_samples, TEST_V_TRUES, pred)
        ], f, ensure_ascii=False)


def predict_probs(model, loader):
    model.eval()
    v_probs, c_probs = [], []
    with torch.no_grad():
        for batch in loader:
            logits_v, logits_c = model(**model_inputs(batch))
            v_probs.append(torch.softmax(logits_v.float(), dim=-1).cpu().numpy())
            if logits_c is not None:
                c_probs.append(torch.softmax(logits_c.float(), dim=-1).cpu().numpy())
    return np.concatenate(v_probs), (np.concatenate(c_probs) if c_probs else None)


def train_one(cfg, seed: int, ckpt_path: str):
    set_seed(seed)
    model = build_model(cfg)
    lam_c, lam_cons = cfg['lambda_conflict'], cfg['lambda_consistency']
    cur_train_loader, cur_dev_loader, cur_test_loader = get_loaders_for(cfg)

    # Hàm loss verdict động hỗ trợ ablation class weighting & label smoothing
    ref_weight = cfg.get('refuted_weight', 1.20)
    cur_weights = torch.tensor([1.0, ref_weight, 1.0], device=DEVICE)
    cur_smoothing = cfg.get('label_smoothing', LABEL_SMOOTHING)
    cur_loss_fn_verdict = nn.CrossEntropyLoss(weight=cur_weights, label_smoothing=cur_smoothing)

    backbone_params = list(model.backbone.parameters())
    head_params = [p for n, p in model.named_parameters() if not n.startswith('backbone.')]
    optimizer = torch.optim.AdamW([
        {'params': backbone_params, 'lr': LR_BACKBONE, 'weight_decay': 1e-2},
        {'params': head_params, 'lr': LR_HEADS, 'weight_decay': 1e-3}
    ], eps=1e-8)
    total_steps = (len(cur_train_loader) // GRAD_ACCUM) * NUM_EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, int(total_steps * 0.1), total_steps)
    scaler = torch.cuda.amp.GradScaler()

    best_f1, best_epoch = -1.0, 0
    print(f"\n========== [{cfg['name']}] seed {seed} | {cfg['label']} ==========")
    for epoch in range(NUM_EPOCHS):
        backbone_trainable = epoch >= HEAD_WARMUP_EPOCHS
        for p in model.backbone.parameters():
            p.requires_grad = backbone_trainable

        model.train()
        sums = {'v': 0.0, 'c': 0.0, 'cons': 0.0, 'kl': 0.0}
        t0 = time.time()
        optimizer.zero_grad()
        for step, batch in enumerate(cur_train_loader):
            inputs = model_inputs(batch)
            labels_v = batch['label_verdict'].to(DEVICE)
            labels_c = batch['label_conflict'].to(DEVICE)

            with torch.cuda.amp.autocast():
                if R_DROP_ALPHA > 0:
                    doubled = {k: torch.cat([v, v], dim=0) for k, v in inputs.items()}
                    labels_v = torch.cat([labels_v, labels_v], dim=0)
                    labels_c = torch.cat([labels_c, labels_c], dim=0)
                    logits_v, logits_c = model(**doubled)
                    logp_a, logp_b = F.log_softmax(logits_v.float(), dim=-1).chunk(2, dim=0)
                    l_kl = 0.5 * (F.kl_div(logp_a, logp_b, reduction='batchmean', log_target=True)
                                  + F.kl_div(logp_b, logp_a, reduction='batchmean', log_target=True))
                else:
                    logits_v, logits_c = model(**inputs)
                    l_kl = torch.zeros((), device=DEVICE)

                zero = torch.zeros((), device=DEVICE)
                l_v = cur_loss_fn_verdict(logits_v, labels_v)
                l_c = loss_fn_conflict(logits_c, labels_c) if (logits_c is not None and lam_c > 0) else zero
                if logits_c is not None and lam_cons > 0:
                    prob_v = torch.softmax(logits_v, dim=-1)
                    prob_c = torch.softmax(logits_c, dim=-1)
                    l_consist = torch.mean(prob_v[:, 0] * (1.0 - prob_c[:, 0]))
                else:
                    l_consist = zero
                loss = (l_v + lam_c * l_c + lam_cons * l_consist + R_DROP_ALPHA * l_kl) / GRAD_ACCUM

            scaler.scale(loss).backward()
            sums['v'] += l_v.item(); sums['c'] += l_c.item(); sums['cons'] += l_consist.item(); sums['kl'] += l_kl.item()

            if (step + 1) % GRAD_ACCUM == 0 or (step + 1) == len(cur_train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                scale_before = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                if scaler.get_scale() >= scale_before:
                    scheduler.step()
                optimizer.zero_grad()

            if (step + 1) % 200 == 0 or (step + 1) == len(cur_train_loader):
                elapsed = time.time() - t0
                speed = (step + 1) / max(elapsed, 1e-5)
                eta = (len(cur_train_loader) - (step + 1)) / max(speed, 1e-5)
                print(f"  [{cfg['name']}] Epoch {epoch+1:02d}/{NUM_EPOCHS:02d} | Step {step+1:04d}/{len(cur_train_loader)} "
                      f"({(step+1)/len(cur_train_loader)*100:4.1f}%) | "
                      f"Loss(V): {sums['v']/(step+1):.3f} | {speed:.1f} b/s | ETA: {eta/60:.1f}m")

        # Đánh giá trên tập Dev để chọn Best Checkpoint
        dev_v_probs, _ = predict_probs(model, cur_dev_loader)
        dev_macro_f1 = f1_score(DEV_V_TRUES, dev_v_probs.argmax(axis=1), average='macro', zero_division=0)
        n = len(cur_train_loader)
        print(f"[{cfg['name']}] Epoch {epoch+1:02d}/{NUM_EPOCHS:02d} [{time.time()-t0:.0f}s] "
              f"({'FULL' if backbone_trainable else 'WARMUP'}) - L(V) {sums['v']/n:.3f} | L(C) {sums['c']/n:.3f} | "
              f"L(Consist) {sums['cons']/n:.3f} | L(KL) {sums['kl']/n:.3f} | Dev Macro-F1 {dev_macro_f1*100:.2f}%")
        if dev_macro_f1 > best_f1:
            best_f1, best_epoch = dev_macro_f1, epoch + 1
            torch.save(model.state_dict(), ckpt_path)
            print(f'  --> 🏆 [BEST DEV] Dev F1 {dev_macro_f1*100:.2f}% (Epoch {best_epoch})')
    return model, best_epoch


# ===================== THỰC THI QUY TRÌNH HUẤN LUYỆN / KIỂM TRA =====================
# 1. Kiểm tra và tự động import Full model từ sar_main_colab nếu có sẵn
has_full = import_full_result_if_available()

print(f"🚀 Bắt đầu xử lý {len(ACTIVE_ABLATIONS)} cấu hình ablation được chọn...\n")

for cfg in ACTIVE_ABLATIONS:
    name = cfg['name']
    probs_path = ablation_probs_path(name)

    # Nếu là Full và đã có kết quả từ sar_main_colab -> Skip ngay
    if name == 'full' and is_ablation_completed('full'):
        print(f"⏩ [full] ĐÃ CÓ KẾT QUẢ TỪ SAR_MAIN_COLAB -> BỎ QUA KHÔNG CẦN HUẤN LUYỆN LẠI!")
        generate_and_print_ablation_table()
        continue

    # Kiểm tra cấu hình đã hoàn thành trước đó chưa
    if is_ablation_completed(name):
        print(f"⏩ [{name}] ĐÃ CÓ KẾT QUẢ TRƯỚC ĐÓ -> BỎ QUA để tiết kiệm GPU quota!")
        generate_and_print_ablation_table()
        continue

    # Huấn luyện nếu chưa có kết quả
    ckpt_path = os.path.join(OUT_DIR, f"{name}_seed{ABLATION_SEED}.pt")
    t_start = time.time()
    model, best_epoch = train_one(cfg, ABLATION_SEED, ckpt_path)
    model.load_state_dict(torch.load(ckpt_path, map_location='cpu'))

    # Đánh giá trên cả Dev (chọn checkpoint) và Test (kết quả cuối cùng)
    cur_train_loader, cur_dev_loader, cur_test_loader = get_loaders_for(cfg)
    dev_v, dev_c = predict_probs(model, cur_dev_loader)
    test_v, test_c = predict_probs(model, cur_test_loader)
    train_mins = (time.time() - t_start) / 60

    # Lưu kết quả
    np.savez(
        probs_path,
        dev_verdict=dev_v,
        dev_conflict=dev_c if dev_c is not None else np.zeros((0, 0)),
        test_verdict=test_v,
        test_conflict=test_c if test_c is not None else np.zeros((0, 0)),
        best_epoch=best_epoch,
        train_minutes=train_mins,
        config=json.dumps(cfg, ensure_ascii=False),
    )
    record_ablation_predictions(name, test_v)
    print(f"\n✅ [{name}] Huấn luyện xong sau {train_mins:.1f} phút | best epoch {best_epoch} | đã lưu {probs_path}")

    # In bảng kết quả cập nhật ngay lập tức
    generate_and_print_ablation_table()

    del model
    gc.collect()
    torch.cuda.empty_cache()
    if not KEEP_CHECKPOINTS and os.path.exists(ckpt_path):
        os.remove(ckpt_path)

print('\n🎉 ĐÃ HOÀN TẤT TẤT CẢ CÁC CẤU HÌNH ĐƯỢC CHỌN!')


## 8. Đánh giá kết quả Ablation trên tập Test (kèm Dev)
Đo lường chi tiết trên tập **Test** (với Dev F1 đối chiếu):
- **Test Macro-F1 & Test Accuracy**
- **Dev Macro-F1** (từ best epoch chọn trên Dev)
- **Per-class F1 trên Test** (Supported, Refuted, NEI)
- **Conflict Slot Accuracy** (trên tập nhãn Refuted của Test)
- **Độ chênh lệch $\Delta$F1** so với mô hình Full trên Test
- **Kiểm định McNemar** ($p$-value so với Full model trên Test)
- Xuất bảng Markdown, CSV, JSON và sinh biểu đồ trực quan hóa.


In [ ]:
# ===================== BẢNG KẾT QUẢ TỔNG THỂ & KIỂM ĐỊNH MCNEMAR TRÊN TẬP TEST =====================
# Hàm generate_and_print_ablation_table đã tự động in và lưu kết quả sau mỗi ablation.
# Chạy cell này để hiển thị lại bảng tổng hợp và trả về biến `results` cho cell vẽ biểu đồ bên dưới.
results = generate_and_print_ablation_table()
print(f"💾 File bảng kết quả đã được lưu tại: {os.path.join(OUT_DIR, 'ablation_table.md')}")
print(f"💾 File CSV kết quả: {os.path.join(OUT_DIR, 'ablation_results.csv')}")
print(f"💾 File JSON kết quả: {RESULTS_JSON_PATH}")


In [ ]:
# ==============================================================================
# TRỰC QUAN HÓA ẤN PHẨM: HEATMAP & SENSITIVITY CURVES
# ==============================================================================
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#333333'
plt.rcParams['axes.linewidth'] = 0.9

# 1. Thu thập dữ liệu: kết hợp kết quả chạy mới nhất và dữ liệu tham chiếu
grid_data = { (r['w'], r['ls']): r for r in REFERENCE_GRID_RESULTS.get(DATASET_NAME, REFERENCE_GRID_RESULTS['vifactcheck']) }

# Nếu có kết quả thực nghiệm mới từ results_json, cập nhật vào grid_data
RESULTS_JSON_PATH = os.path.join(OUT_DIR, 'ablation_results.json')
if os.path.exists(RESULTS_JSON_PATH):
    try:
        with open(RESULTS_JSON_PATH, 'r') as fp:
            run_res = json.load(fp)
        for name, m in run_res.items():
            # parse w and ls from name e.g. w_1.20_ls_0.10
            parts = name.split('_')
            if len(parts) >= 4 and parts[0] == 'w' and parts[2] == 'ls':
                w_val = float(parts[1])
                ls_val = float(parts[3])
                grid_data[(w_val, ls_val)] = {
                    'w': w_val, 'ls': ls_val,
                    'dev_f1': m.get('dev_macro_f1', 0)*100,
                    'test_f1': m.get('test_macro_f1', 0)*100,
                    'sup_f1': m.get('per_class_f1', {}).get('Supported', 0)*100,
                    'ref_f1': m.get('per_class_f1', {}).get('Refuted', 0)*100,
                    'nei_f1': m.get('per_class_f1', {}).get('NEI', 0)*100,
                }
        print(f"✓ Đã đồng bộ kết quả thực tế từ {RESULTS_JSON_PATH}")
    except Exception as e:
        print("Lỗi đọc kết quả mới:", e)

# 2. Chuẩn bị ma trận Heatmap 2D
w_list = [1.00, 1.10, 1.20, 1.30]
ls_list = [0.00, 0.05, 0.10, 0.15]
matrix_dev = np.zeros((len(w_list), len(ls_list)))

for i, w in enumerate(w_list):
    for j, ls in enumerate(ls_list):
        if (w, ls) in grid_data:
            matrix_dev[i, j] = grid_data[(w, ls)]['dev_f1']

# 3. Vẽ 2 đồ thị ấn phẩm
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6), dpi=300)

# (a) Heatmap 2D
im = ax1.imshow(matrix_dev, cmap='YlGnBu', aspect='auto', origin='lower')
ax1.set_xticks(range(len(ls_list)))
ax1.set_xticklabels([f'{v:.2f}' for v in ls_list], fontsize=11)
ax1.set_yticks(range(len(w_list)))
ax1.set_yticklabels([f'{v:.2f}' for v in w_list], fontsize=11)
ax1.set_xlabel(r'Label Smoothing ($\epsilon$)', fontsize=12, fontweight='bold')
ax1.set_ylabel(r'Refuted Class Weight ($w_{\mathrm{refuted}}$)', fontsize=12, fontweight='bold')
ax1.set_title('(a) Dev Macro-F1 (%) Heatmap Grid Search', fontsize=13, fontweight='bold', pad=12)

# Thêm nhãn giá trị và khoanh đỏ điểm tối ưu (w=1.20, ls=0.10)
for i in range(len(w_list)):
    for j in range(len(ls_list)):
        val = matrix_dev[i, j]
        is_opt = (w_list[i] == 1.20 and ls_list[j] == 0.10)
        txt = f'{val:.2f}%' + ('\n★ Opt' if is_opt else '')
        color = 'white' if val > np.median(matrix_dev) else '#222222'
        weight = 'bold' if is_opt else 'normal'
        ax1.text(j, i, txt, ha='center', va='center', color=color, fontsize=10, fontweight=weight)
        if is_opt:
            rect = plt.Rectangle((j - 0.48, i - 0.48), 0.96, 0.96, fill=False, edgecolor='#d90429', lw=2.5, ls='--')
            ax1.add_patch(rect)

cbar = fig.colorbar(im, ax=ax1, fraction=0.046, pad=0.04)
cbar.set_label('Dev Macro-F1 (%)', fontsize=11)

# (b) Per-Class Sensitivity Curve (Fix ls = 0.10, vary w_refuted from 1.00 to 1.50)
curve_w = [1.00, 1.10, 1.20, 1.30, 1.50]
sup_f1s = []
ref_f1s = []
nei_f1s = []
macro_f1s = []

for w in curve_w:
    if (w, 0.10) in grid_data:
        d = grid_data[(w, 0.10)]
        sup_f1s.append(d['sup_f1'])
        ref_f1s.append(d['ref_f1'])
        nei_f1s.append(d['nei_f1'])
        macro_f1s.append(d['test_f1'])
    else:
        sup_f1s.append(np.nan)
        ref_f1s.append(np.nan)
        nei_f1s.append(np.nan)
        macro_f1s.append(np.nan)

ax2.plot(curve_w, ref_f1s, 's--', color='#d90429', lw=2.2, label='Refuted F1 (Cost sensitivity)', markersize=7)
ax2.plot(curve_w, sup_f1s, 'o-', color='#1d3557', lw=2.2, label='Supported F1 (Precision balance)', markersize=7)
ax2.plot(curve_w, nei_f1s, '^-.', color='#457b9d', lw=1.8, label='NEI F1', markersize=6)
ax2.plot(curve_w, macro_f1s, 'D-', color='#2a9d8f', lw=2.6, label='Test Macro-F1 (Overall)', markersize=8)

ax2.axvline(x=1.20, color='#d90429', linestyle=':', lw=1.8, alpha=0.8, label='Selected ($w=1.20$)')
ax2.set_xlabel(r'Refuted Class Weight ($w_{\mathrm{refuted}}$)', fontsize=12, fontweight='bold')
ax2.set_ylabel('Test F1 Score (%)', fontsize=12, fontweight='bold')
ax2.set_title(r'(b) Per-Class F1 Trade-off with Fixed $\epsilon=0.10$', fontsize=13, fontweight='bold', pad=12)
ax2.set_xticks(curve_w)
ax2.set_xticklabels([f'{w:.2f}' for w in curve_w], fontsize=11)
ax2.grid(True, linestyle='--', alpha=0.5)
ax2.legend(fontsize=10, loc='lower left')

plt.tight_layout()
fig_out_path = os.path.join(OUT_DIR, f'figure_verdict_loss_tuning_{DATASET_NAME}.png')
plt.savefig(fig_out_path, dpi=300, bbox_inches='tight')
print(f"📊 Đã lưu đồ thị ấn phẩm tại: {fig_out_path}")
plt.show()
